# SignX — Full Training Pipeline for Omani Sign Language

**Paper:** [SignX (arXiv:2504.16315v3)](https://arxiv.org/abs/2504.16315)
**Server:** 4 × NVIDIA GPU, 24 GB VRAM each (96 GB total)
**Mixed precision:** fp16 AMP enabled by default
**Multi-GPU:** DataParallel across all 4 GPUs

---

## How to use this notebook

Run cells **top to bottom**. Never skip Part 0 — it sets paths and seeds.

| Symbol | Meaning |
|--------|---------|
| 📌 | Configuration you may need to edit |
| ⚠️ | Important warning |
| 💡 | Tip or explanation |
| [OPTIONAL] | Requires extra GPU memory or packages |

## Files needed on the training server

You need the **entire `OSL_SignX/` folder**, not just this notebook.

```
OSL_SignX/
├── signx/           ← Python package (all model/training code)
├── configs/         ← YAML configs for each stage
├── data/            ← gloss_vocab.txt, Words.txt, Sentences.txt, sentence_glosses.txt
├── requirements.txt ← pip dependencies
└── notebooks/       ← this notebook
```

The dataset lives separately (set `DATASET_ROOT` in Part 0 Cell 1).
Copy or mount the dataset so the server can reach it.


---
## Part 0 — Environment Setup

Before anything else we:
1. Point Python at the `signx` package
2. Configure the dataset path
3. Verify the GPU setup
4. Install dependencies
5. Set random seeds for reproducibility


In [ ]:
# 📌 Edit these two paths if needed
import sys, os

PROJECT_ROOT = '/home/ahmed/Models/OSL_SignX'
DATASET_ROOT = '/mnt/c/Users/MOBPC/Downloads/FYP/FYPproject/OSL_Dataset'

# Make `import signx` work from anywhere in this notebook
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Switch working directory so relative paths in configs resolve correctly
os.chdir(PROJECT_ROOT)

# Verify dataset
if not os.path.exists(DATASET_ROOT):
    print(f'⚠️  Dataset not found at:\n    {DATASET_ROOT}')
    print('Edit DATASET_ROOT above to point to your dataset directory.')
else:
    print(f'✓  Dataset root: {DATASET_ROOT}')
print(f'✓  Project root: {PROJECT_ROOT}')


In [ ]:
# GPU inventory — verifies 4×24 GB are visible
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPUs    : {torch.cuda.device_count()}')
print()

for i in range(torch.cuda.device_count()):
    p   = torch.cuda.get_device_properties(i)
    mem = p.total_memory / 1024**3
    print(f'  GPU {i}: {p.name:<35} {mem:.1f} GB  CC {p.major}.{p.minor}')

assert torch.cuda.is_available(), '⚠️  No CUDA GPU found — cannot train.'

DEVICE      = 'cuda'
N_GPUS      = torch.cuda.device_count()
TOTAL_VRAM  = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(N_GPUS)
) / 1024**3
print(f'\n✓  {N_GPUS} GPU(s)  total VRAM = {TOTAL_VRAM:.0f} GB')
# Expected: 4 GPUs, 96 GB total


In [ ]:
# Install / verify dependencies
# uv is used on this server; fall back to pip if uv is not present
import subprocess, shutil

req = os.path.join(PROJECT_ROOT, 'requirements.txt')
installer = ['uv', 'pip', 'install', '-q', '-r', req] if shutil.which('uv') \
            else ['pip', 'install', '-q', '-r', req]
result = subprocess.run(installer, capture_output=True, text=True)
if result.returncode != 0:
    print('pip stderr:', result.stderr[:800])
else:
    print('✓  Dependencies installed / already satisfied')


In [ ]:
# Reproducibility — must be called before any model or data initialisation
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# deterministic=True slows down conv ops; keep False for speed
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True   # auto-tune convolutions for fixed input sizes
print(f'✓  Seeds fixed to {SEED}  |  cudnn.benchmark=True')


In [ ]:
# Full environment summary table
import importlib, platform

print(f'Python   : {platform.python_version()}')
print(f'Platform : {platform.platform()}')
print()
pkgs = ['torch', 'torchvision', 'numpy', 'scipy', 'mediapipe',
        'decord', 'cv2', 'PIL', 'omegaconf', 'tqdm',
        'editdistance', 'sacrebleu', 'einops', 'timm']
print(f'  {"Package":<18} {"Version"}')
print('  ' + '-'*30)
for pkg in pkgs:
    try:
        m   = importlib.import_module(pkg)
        ver = getattr(m, '__version__', 'installed')
        print(f'  {pkg:<18} {ver}')
    except ImportError:
        print(f'  {pkg:<18} ✗ NOT INSTALLED')


In [ ]:
# Matplotlib — professional publication style
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'figure.dpi'    : 150,
    'figure.figsize': (9, 5),
    'axes.grid'     : True,
    'grid.alpha'    : 0.3,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'     : 11,
})
import time
FIG_DIR = os.path.join(PROJECT_ROOT, 'notebooks', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'✓  Figures → {FIG_DIR}')


---
## Part 1 — Dataset Exploration & Validation

### What we're checking
- **Video counts** per split (train / dev / test / sentences)
- **Vocabulary** — 802 entries: 1 blank + 801 Arabic glosses
- **Video durations** — important for choosing `max_frames`
- **Class distribution** — imbalanced datasets hurt CTC training
- **Thumbnail strips** — quick sanity check that files decode correctly

### Dataset structure expected on disk
```
DATASET_ROOT/
├── final_split/
│   ├── train/rgb/  *.mp4   (word-level, 1 gloss per video)
│   ├── dev/rgb/    *.mp4
│   └── test/rgb/   *.mp4
└── OSL-Sentences/
    └── rgb_format/ *.mp4   (sentence-level, used in Stage 3)
```


In [ ]:
# Load the paths config (merges paths.yaml → default.yaml)
from signx.utils import load_config
from omegaconf import OmegaConf

paths_cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'paths.yaml'))
print('Resolved paths:')
print(OmegaConf.to_yaml(paths_cfg))


In [ ]:
# Count MP4 files in every split
import glob

WORD_SPLIT = os.path.join(DATASET_ROOT, 'final_split')
SENT_DIR   = os.path.join(DATASET_ROOT, 'OSL-Sentences', 'rgb_format')

split_counts = {}
for split in ['train', 'dev', 'test']:
    mp4s = glob.glob(os.path.join(WORD_SPLIT, split, 'rgb', '**', '*.mp4'), recursive=True)
    split_counts[split] = len(mp4s)

sent_mp4s = glob.glob(os.path.join(SENT_DIR, '**', '*.mp4'), recursive=True)
split_counts['sentences'] = len(sent_mp4s)

print(f'  {"Split":<12} {"Videos":>8}')
print('  ' + '-'*22)
for k, v in split_counts.items():
    print(f'  {k:<12} {v:>8}')

# Warn if any split is empty
for k, v in split_counts.items():
    if v == 0:
        print(f'\n⚠️  {k} split is empty — check DATASET_ROOT and directory structure.')


In [ ]:
# Load and inspect the vocabulary
from signx.data.vocab import GlossVocab

VOCAB_FILE = os.path.join(PROJECT_ROOT, 'data', 'gloss_vocab.txt')
vocab      = GlossVocab.from_file(VOCAB_FILE)

print(f'Vocab size : {len(vocab)}')
print(f'Blank ID   : 0  ("<blank>")')
print()
print('First 10 entries:')
for i in range(min(10, len(vocab))):
    g = vocab.id2gloss[i] if hasattr(vocab, 'id2gloss') else vocab.decode([i], strip_blank=False)[0]
    print(f'  {i:04d}  {g}')
print('...')


In [ ]:
# Words.txt — one Arabic word per line
WORDS_FILE = os.path.join(PROJECT_ROOT, 'data', 'Words.txt')
with open(WORDS_FILE, encoding='utf-8') as f:
    words = [l.strip() for l in f if l.strip()]
print(f'Words.txt  : {len(words)} entries')
print('Sample     :', words[:5])

# Sentences.txt
SENT_FILE = os.path.join(PROJECT_ROOT, 'data', 'Sentences.txt')
with open(SENT_FILE, encoding='utf-8') as f:
    sentences = [l.strip() for l in f if l.strip()]
print(f'Sentences  : {len(sentences)} entries')

# sentence_glosses.txt
SG_FILE = os.path.join(PROJECT_ROOT, 'data', 'sentence_glosses.txt')
sg_lens = []
with open(SG_FILE, encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) > 1:
            sg_lens.append(len(parts) - 1)  # subtract sentence-ID token
print(f'Sent glosses: {len(sg_lens)} sequences')
if sg_lens:
    print(f'Gloss seq length — min:{min(sg_lens)}  max:{max(sg_lens)}  mean:{np.mean(sg_lens):.1f}')


In [ ]:
# Video duration analysis
# We sample up to 300 videos from train — enough for a reliable histogram
import cv2

train_mp4s = glob.glob(
    os.path.join(WORD_SPLIT, 'train', 'rgb', '**', '*.mp4'), recursive=True
)
sample_for_dur = random.sample(train_mp4s, min(300, len(train_mp4s)))

durations, frame_counts = [], []
for p in sample_for_dur:
    cap = cv2.VideoCapture(p)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n   = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    if fps > 0 and n > 0:
        durations.append(n / fps)
        frame_counts.append(int(n))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(durations,    bins=30, color='steelblue', edgecolor='white')
ax1.set_xlabel('Duration (s)')
ax1.set_ylabel('Video count')
ax1.set_title(f'Video Duration (n={len(durations)})')

ax2.hist(frame_counts, bins=30, color='coral',     edgecolor='white')
ax2.axvline(256, color='crimson', linestyle='--', label='max_frames=256')
ax2.set_xlabel('Frame count')
ax2.set_ylabel('Video count')
ax2.set_title('Frame Counts')
ax2.legend()

plt.suptitle('OSL Word Dataset — Temporal Statistics (train sample)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'duration_hist.png'), dpi=150, bbox_inches='tight')
plt.show()

pct_trunc = 100 * sum(1 for n in frame_counts if n > 256) / len(frame_counts)
print(f'Mean duration : {np.mean(durations):.2f}s')
print(f'Max frames    : {max(frame_counts)}')
print(f'Truncated (>256 frames): {pct_trunc:.1f}%  — these will be clipped to max_frames=256')


In [ ]:
# Thumbnail strip — first, middle, last frame of 5 random videos
sample_vids = random.sample(train_mp4s, min(5, len(train_mp4s)))

def read_frame_at(path, frac):
    cap = cv2.VideoCapture(path)
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frac * (n - 1)))
    ret, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else np.zeros((224, 224, 3), dtype=np.uint8)

fig, axes = plt.subplots(len(sample_vids), 3, figsize=(9, 3 * len(sample_vids)))
for row, (ax_row, vp) in enumerate(zip(axes, sample_vids)):
    for ax, frac, title in zip(ax_row, [0.0, 0.5, 1.0], ['First', 'Middle', 'Last']):
        ax.imshow(read_frame_at(vp, frac))
        ax.set_title(f'{title}\n{os.path.basename(vp)[:22]}', fontsize=8)
        ax.axis('off')
plt.suptitle('Thumbnail Strip (5 random train videos)', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'thumbnails.png'), dpi=150, bbox_inches='tight')
plt.show()

print('✓  Visual check passed — videos decode correctly')


---
## Part 2 — Pose Extraction Demo (All 5 Methods)

SignX builds a **1959-dim pose vector** per frame by concatenating five complementary methods:

| Method | Dims | What it captures | Hardware |
|--------|------|-----------------|----------|
| MediaPipe | 258 | 3-D body / hand / face joints | CPU |
| DWPose | 399 | 2-D keypoints + confidence | GPU ~2 GB |
| SMPLer-X | 432 | 3-D body mesh (SMPL params) | GPU ~8 GB |
| PrimeDepth | 480 | Monocular depth features | GPU ~4 GB |
| Sapiens | 390 | Body-part segmentation | GPU ~6 GB |

**MediaPipe is the default** (backend = `mediapipe`, 258-dim).
The other four are optional and wrapped in `try/except`.
If `.pkl` pre-extracted files exist in the dataset, they are loaded instead.

💡 For faster training, pre-extract all pose features once and set `pose.backend = precomputed`.


In [ ]:
# Select a representative demo video (middle of the first train sample)
DEMO_VIDEO = sample_vids[0]
print(f'Demo video: {DEMO_VIDEO}')

cap = cv2.VideoCapture(DEMO_VIDEO)
n_demo = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_demo = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, n_demo // 2)
ret, rep_frame = cap.read()
cap.release()

if ret:
    rep_rgb = cv2.cvtColor(rep_frame, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(rep_rgb)
    ax.set_title(f'{n_demo} frames @ {fps_demo:.0f} fps')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Resolution: {rep_frame.shape[1]}×{rep_frame.shape[0]}')


### Method 1 — MediaPipe (258-dim, CPU)

In [ ]:
# MediaPipe decomposes the pose into three groups:
#   Body    : 33 landmarks × 4 values (x, y, z, visibility) = 132 dims
#   Left hand : 21 landmarks × 3 values (x, y, z)            =  63 dims
#   Right hand: 21 landmarks × 3 values                      =  63 dims
#   ─────────────────────────────────────────────────────────────────────
#   Total   : 258 dims

from signx.pose.mediapipe_extractor import MediaPipePoseExtractor

mp_extractor = MediaPipePoseExtractor()
t0 = time.time()
mp_pose = mp_extractor.extract(DEMO_VIDEO)   # (T, 258) float32
elapsed = time.time() - t0

print(f'Output shape : {tuple(mp_pose.shape)}')
print(f'Extraction   : {elapsed:.2f}s  →  {mp_pose.shape[0]/elapsed:.1f} effective fps')
print(f'Min / max    : {mp_pose.min():.4f} / {mp_pose.max():.4f}')
print(f'Non-zero pct : {(mp_pose != 0).float().mean()*100:.1f}%')


In [ ]:
# Heatmap: each row = one pose dimension, each column = one frame
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(mp_pose.numpy().T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Frame index')
ax.set_ylabel('Pose dimension')
ax.axhline(132, color='yellow', lw=0.8, linestyle='--', label='body | left-hand')
ax.axhline(195, color='cyan',   lw=0.8, linestyle='--', label='left-hand | right-hand')
ax.set_title('MediaPipe Pose Sequence Heatmap [T × 258]')
ax.legend(fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'mediapipe_heatmap.png'), dpi=150)
plt.show()


In [ ]:
# Skeleton overlay — annotate one frame with all landmarks
import mediapipe as mp_lib
mp_holistic = mp_lib.solutions.holistic
mp_drawing  = mp_lib.solutions.drawing_utils
mp_styles   = mp_lib.solutions.drawing_styles

annotated = rep_frame.copy()
with mp_holistic.Holistic(static_image_mode=True, min_detection_confidence=0.5) as hol:
    results = hol.process(rep_rgb)
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            annotated, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style())
    for hand_lm in [results.left_hand_landmarks, results.right_hand_landmarks]:
        if hand_lm:
            mp_drawing.draw_landmarks(
                annotated, hand_lm, mp_holistic.HAND_CONNECTIONS,
                mp_styles.get_default_hand_landmarks_style(),
                mp_styles.get_default_hand_connections_style())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(rep_rgb)
axes[0].set_title('Original frame')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('MediaPipe Holistic Overlay')
axes[1].axis('off')
plt.suptitle('Pose Skeleton Visualisation')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'mediapipe_overlay.png'), dpi=150)
plt.show()


### Methods 2–5 — DWPose / SMPLer-X / PrimeDepth / Sapiens [OPTIONAL]

In [ ]:
# [OPTIONAL] DWPose — 399-dim
# 18 body × (x,y,conf) + 21×2 hand × (x,y,conf) + 68 face × (x,y,conf) = 399
# Install: see signx/pose/dwpose_extractor.py for requirements
try:
    from signx.pose.dwpose_extractor import DWPoseExtractor
    dw = DWPoseExtractor()
    dw_pose = dw.extract(DEMO_VIDEO)
    print(f'DWPose shape : {tuple(dw_pose.shape)}   (expected T×399)')
except Exception as e:
    print(f'DWPose not available : {e}')
    print('Expected output : (T, 399)')
    dw_pose = None


In [ ]:
# [OPTIONAL] SMPLer-X — 432-dim  (~8 GB GPU required)
# SMPL body pose (72) + shape (10) + expression (10) + hands (90) + ... = 432
try:
    from signx.pose.smplerx_extractor import SMPLerXExtractor
    sx = SMPLerXExtractor()
    sx_pose = sx.extract(DEMO_VIDEO)
    print(f'SMPLer-X shape: {tuple(sx_pose.shape)}  (expected T×432)')
except Exception as e:
    print(f'SMPLer-X not available: {e}')
    sx_pose = None


In [ ]:
# [OPTIONAL] PrimeDepth — 480-dim  (~4 GB GPU required)
# Monocular depth map downsampled and flattened per frame
try:
    from signx.pose.primedepth_extractor import PrimeDepthExtractor
    pd_ext = PrimeDepthExtractor()
    pd_pose = pd_ext.extract(DEMO_VIDEO)
    print(f'PrimeDepth shape: {tuple(pd_pose.shape)}  (expected T×480)')

    # Visualise depth map for the representative frame
    import importlib
    if importlib.util.find_spec('matplotlib'):
        depth_map = pd_pose[n_demo // 2].numpy().reshape(-1, 1)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(depth_map, cmap='jet', aspect='auto')
        ax.set_title('PrimeDepth — frame depth features')
        plt.tight_layout(); plt.show()
except Exception as e:
    print(f'PrimeDepth not available: {e}')
    pd_pose = None


In [ ]:
# [OPTIONAL] Sapiens — 390-dim  (~6 GB GPU required)
# Fine-grained body-part segmentation features
try:
    from signx.pose.sapiens_extractor import SapiensExtractor
    sap = SapiensExtractor()
    sap_pose = sap.extract(DEMO_VIDEO)
    print(f'Sapiens shape : {tuple(sap_pose.shape)}  (expected T×390)')
except Exception as e:
    print(f'Sapiens not available: {e}')
    sap_pose = None


In [ ]:
# Comparison table — summarise all 5 methods
available = {'MediaPipe': True, 'DWPose': dw_pose is not None,
             'SMPLer-X': sx_pose is not None, 'PrimeDepth': pd_pose is not None,
             'Sapiens': sap_pose is not None}
table = [
    ('MediaPipe',  258, '3-D body/hand/face joints',    '~30 fps', 'CPU only'),
    ('DWPose',     399, '2-D keypoints + confidence',   '~15 fps', '~2 GB'),
    ('SMPLer-X',   432, '3-D body mesh params (SMPL)',  '~3  fps', '~8 GB'),
    ('PrimeDepth', 480, 'Monocular depth features',     '~10 fps', '~4 GB'),
    ('Sapiens',    390, 'Body-part segmentation feats', '~8  fps', '~6 GB'),
]
print(f'  {"Method":<12} {"Dims":>5}  {"What it captures":<35} {"Speed":>9}  {"VRAM":<10} {"Avail"}')
print('  ' + '-'*85)
total = 0
for name, dims, desc, spd, mem in table:
    total += dims
    mark = '✓' if available[name] else '✗'
    print(f'  {name:<12} {dims:>5}  {desc:<35} {spd:>9}  {mem:<10} {mark}')
print(f'  {"Concatenated":<12} {total:>5}')


---
## Part 3 — Data Pipeline Setup

### How data flows through training
```
MP4 file → decord VideoReader → float32 (T,H,W,3) → VideoTransform
         → (T, 3, 224, 224) normalised tensor → collate_video_batch
         → padded batch (B, T_max, 3, 224, 224) → DataLoader → GPU
```

### Key parameters
- `max_frames = 256` — videos longer than this are centre-cropped in time
- `batch_size` — see Part 5/6/7 for per-stage values
- `num_workers = 8` (2 per GPU) — tune up if CPU is idle during training
- `pin_memory = True` — speeds up host→GPU transfers


In [ ]:
from signx.data.dataset   import OSLWordDataset
from signx.data.collate   import collate_video_batch
from signx.data.transforms import VideoTransform, VideoTransformConfig
from torch.utils.data      import DataLoader

stage1_cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'stage1_pose2gloss.yaml'))

train_transform = VideoTransform(VideoTransformConfig(image_size=224, train=True))
val_transform   = VideoTransform(VideoTransformConfig(image_size=224, train=False))

try:
    train_ds = OSLWordDataset(WORD_SPLIT, 'train', vocab,
                              transform=train_transform,
                              max_frames=stage1_cfg.video.max_frames)
    val_ds   = OSLWordDataset(WORD_SPLIT, 'dev',   vocab,
                              transform=val_transform,
                              max_frames=stage1_cfg.video.max_frames)
    test_ds  = OSLWordDataset(WORD_SPLIT, 'test',  vocab,
                              transform=val_transform,
                              max_frames=stage1_cfg.video.max_frames)
    print(f'✓  train : {len(train_ds):>6} samples')
    print(f'✓  dev   : {len(val_ds):>6} samples')
    print(f'✓  test  : {len(test_ds):>6} samples')
except Exception as e:
    print(f'⚠️  Dataset init error: {e}')
    print('Check DATASET_ROOT and final_split/ directory structure.')
    train_ds = val_ds = test_ds = None


In [ ]:
# Inspect one sample — shows what the model actually receives
if train_ds:
    s = train_ds[0]
    print('Keys         :', list(s.keys()))
    print(f'video        : {s["video"].shape}   dtype={s["video"].dtype}')
    print(f'gloss_ids    : {s["gloss_ids"].tolist()}')
    print(f'video_length : {s["video_length"]}')
    print(f'gloss_length : {s["gloss_length"]}')
    print(f'item_id      : {s["item_id"]}')

    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    idxs = [0, s['video_length']//3, 2*s['video_length']//3, s['video_length']-1]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    for ax, idx in zip(axes, idxs):
        frame = (s['video'][idx] * std + mean).clamp(0,1).permute(1,2,0).numpy()
        ax.imshow(frame)
        ax.set_title(f'Frame {idx}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Un-normalised frames from one training sample')
    plt.tight_layout(); plt.show()


In [ ]:
# Test the collate function with a mini batch of 4
# collate_video_batch pads videos to the longest sequence in the batch
if train_ds:
    probe_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                              num_workers=0, collate_fn=collate_video_batch)
    batch = next(iter(probe_loader))

    print('Collated batch keys   :', list(batch.keys()))
    print(f'videos       : {batch["videos"].shape}   (B, T_max, C, H, W)')
    print(f'video_lengths: {batch["video_lengths"].tolist()}')
    print(f'glosses      : {batch["glosses"].shape}')
    print(f'gloss_lengths: {batch["gloss_lengths"].tolist()}')
    mem_mb = batch['videos'].element_size() * batch['videos'].nelement() / 1024**2
    print(f'\nRaw video tensor : {mem_mb:.1f} MB per batch of 4')
    print(f'On GPU (fp16)    : {mem_mb/2:.1f} MB   ← AMP halves this')
    del probe_loader


In [ ]:
# DataLoader throughput benchmark
# num_workers=8 matches default.yaml (2 per GPU × 4 GPUs)
if train_ds:
    bench_loader = DataLoader(
        train_ds, batch_size=8, shuffle=True,
        num_workers=8, pin_memory=True, collate_fn=collate_video_batch,
        persistent_workers=True,
    )
    t0, n, nb = time.time(), 0, min(20, len(bench_loader))
    for i, b in enumerate(bench_loader):
        n += b['videos'].shape[0]
        if i >= nb-1: break
    elapsed = time.time() - t0
    print(f'Throughput: {n/elapsed:.1f} samples/s  |  {nb/elapsed:.1f} batches/s')
    print(f'Est. time per epoch (train): {len(train_ds)/(n/elapsed)/60:.1f} min  (wall-clock, CPU only)')
    del bench_loader


---
## Part 4 — Model Architecture Inspection

### Three-stage pipeline

```
┌─ Stage 1: Pose2Gloss ──────────────────────────────────────────────────┐
│  MediaPipe pose (T×258)                                                │
│       → PoseFusionEncoder [Multi-head self-attention, 2 layers]        │
│       → latent (T×512)                                                 │
│       → CodeBookDecoder [Transformer decoder, 4 layers]               │
│       → gloss logits (L×802)                                           │
└────────────────────────────────────────────────────────────────────────┘

┌─ Stage 2: Video2Pose ──────────────────────────────────────────────────┐
│  RGB video (T×3×224×224)                                               │
│       → ViTFrameBackbone [vit_base_patch16_224, per-frame]             │
│       → (T×768)                                                        │
│       → Linear projection → (T×512)   [matches Stage 1 latent]        │
│  Loss: MSE( pred_latent, gt_latent_from_stage1 )                       │
└────────────────────────────────────────────────────────────────────────┘

┌─ Stage 3: CSLR ─────────────────────────────────────────────────────────┐
│  Video → Stage2 → latent (T×512)                                        │
│       → FeaturePruning (512→256)                                        │
│       → ResNet1D + TemporalConv                                         │
│       → BiLSTM [2 layers, hidden=256, bidirectional → 512]              │
│       → Transformer [4-layer encoder-decoder, d_model=256]              │
│       → CTC head (T'×802)  ←── CTC loss                                │
│       → Decoder (L×802)    ←── CE loss + KD loss                       │
│  Beam search (beam=8) at inference time                                 │
└─────────────────────────────────────────────────────────────────────────┘
```

All three stages use **DataParallel** (4 GPUs) and **fp16 AMP**.


In [ ]:
from signx.models.signx_model import Stage1Model, Stage2Model, Stage3Model

stage2_cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'stage2_video2pose.yaml'))
stage3_cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'stage3_cslr.yaml'))

VOCAB_SIZE = len(vocab)           # 802
LATENT_DIM = int(stage1_cfg.model.latent_dim)   # 512
POSE_DIM   = int(stage1_cfg.model.pose_input_dim)  # 258

print(f'vocab_size  = {VOCAB_SIZE}')
print(f'latent_dim  = {LATENT_DIM}')
print(f'pose_dim    = {POSE_DIM}')


In [ ]:
def make_stage1():
    return Stage1Model(
        pose_input_dim    = POSE_DIM,
        latent_dim        = LATENT_DIM,
        codebook_size     = int(stage1_cfg.model.codebook_size),
        codebook_dim      = int(stage1_cfg.model.codebook_dim),
        decoder_layers    = int(stage1_cfg.model.decoder_layers),
        num_heads         = int(stage1_cfg.model.num_fusion_heads),
        vocab_size        = VOCAB_SIZE,
        num_fusion_layers = int(stage1_cfg.model.num_fusion_layers),
        dropout           = float(stage1_cfg.model.dropout),
    )

def make_stage2(s1_model=None):
    return Stage2Model(
        latent_dim     = LATENT_DIM,
        vit_name       = str(stage2_cfg.model.vit_name),
        vit_pretrained = bool(stage2_cfg.model.vit_pretrained),
        stage1_model   = s1_model,
    )

def make_stage3():
    return Stage3Model(
        latent_dim         = LATENT_DIM,
        pruned_dim         = int(stage3_cfg.model.pruned_dim),
        tconv_channels     = list(stage3_cfg.model.tconv_channels),
        tconv_kernels      = list(stage3_cfg.model.tconv_kernels),
        tconv_strides      = list(stage3_cfg.model.tconv_strides),
        lstm_hidden        = int(stage3_cfg.model.lstm_hidden),
        lstm_layers        = int(stage3_cfg.model.lstm_layers),
        bidirectional      = bool(stage3_cfg.model.lstm_bidirectional),
        transformer_layers = int(stage3_cfg.model.transformer_layers),
        transformer_dim    = int(stage3_cfg.model.transformer_dim),
        transformer_heads  = int(stage3_cfg.model.transformer_heads),
        transformer_ffn    = int(stage3_cfg.model.transformer_ffn),
        vocab_size         = VOCAB_SIZE,
        dropout_attn       = float(stage3_cfg.model.dropout_attn),
        dropout_relu       = float(stage3_cfg.model.dropout_relu),
        dropout_res        = float(stage3_cfg.model.dropout_res),
        max_target_len     = int(stage3_cfg.decode.max_len),
    )

stage1_model = make_stage1().to(DEVICE)
stage2_model = make_stage2(stage1_model).to(DEVICE)
stage3_model = make_stage3().to(DEVICE)

p1 = sum(p.numel() for p in stage1_model.parameters())
p2 = sum(p.numel() for p in stage2_model.parameters())
p3 = sum(p.numel() for p in stage3_model.parameters())

print(f'  {"Stage":<10} {"Params":>15}  Notes')
print('  ' + '-'*55)
print(f'  {"Stage 1":<10} {p1:>15,}  PoseFusionEncoder + CodeBookDecoder')
print(f'  {"Stage 2":<10} {p2:>15,}  ViT-B/16 + projection head')
print(f'  {"Stage 3":<10} {p3:>15,}  ResNet1D + TemporalConv + BiLSTM + Transformer')
print(f'  {"TOTAL":<10} {p1+p2+p3:>15,}')


In [ ]:
# Forward pass sanity check with dummy data
B, T, L = 2, 30, 8
dummy_pose   = torch.randn(B, T, POSE_DIM, device=DEVICE)
dummy_target = torch.randint(1, VOCAB_SIZE, (B, L), device=DEVICE)   # avoid blank=0

stage1_model.eval()
with torch.no_grad():
    out1 = stage1_model(dummy_pose, dummy_target)

print('Stage 1 output shapes:')
for k, v in out1.items():
    print(f'  {k:<12}: {tuple(v.shape)}')

# Stage 3 uses latent from Stage 1
with torch.no_grad():
    lat     = out1['latent']                           # (B, T, 512)
    lengths = torch.full((B,), T, dtype=torch.long, device=DEVICE)
    out3    = stage3_model(lat, target=dummy_target, lengths=lengths)

print('\nStage 3 output shapes:')
for k, v in out3.items():
    if hasattr(v, 'shape'):
        print(f'  {k:<12}: {tuple(v.shape)}')
    else:
        print(f'  {k:<12}: {v}')

print('\n✓  Forward passes clean')


---
## GPU Memory Helper

We define a reusable function `gpu_mem()` used throughout training to
verify we never approach OOM before starting each stage.


In [ ]:
def gpu_mem():
    """Print free / total VRAM for every GPU."""
    if not torch.cuda.is_available():
        print('No GPU'); return
    torch.cuda.synchronize()
    for i in range(torch.cuda.device_count()):
        total    = torch.cuda.get_device_properties(i).total_memory / 1024**3
        reserved = torch.cuda.memory_reserved(i)  / 1024**3
        alloc    = torch.cuda.memory_allocated(i) / 1024**3
        free     = total - reserved
        bar      = '█' * int(reserved/total*20) + '░' * (20 - int(reserved/total*20))
        print(f'  GPU {i}: [{bar}] {reserved:.1f}/{total:.1f} GB  '
              f'allocated={alloc:.1f} GB  free≈{free:.1f} GB')

gpu_mem()


---
## Part 5 — Stage 1 Training: Pose2Gloss

### Goal
Train PoseFusionEncoder + CodeBookDecoder so that **MediaPipe pose features
→ correct Arabic gloss tokens**.  This establishes a semantically rich 512-dim
latent space that Stage 2 will later learn to replicate from raw RGB.

### Multi-GPU strategy
- `multi_gpu = true` → DataParallel splits each batch across 4 GPUs
- `use_amp = true`   → fp16 reduces VRAM ~50 % and speeds up ~2×
- Effective batch: 128 = 32 per GPU × 4 GPUs

### Loss functions
| Loss | Weight | Purpose |
|------|--------|---------|
| `text_loss` | λ=1 | Cross-entropy on predicted gloss tokens |
| `word_match_loss` | λ=1 | Padded CE with label smoothing 0.1 |
| `contrastive_loss` | λ=1 | InfoNCE between pose & gloss embeddings (τ=0.07) |

### Config summary
- Epochs: 800  |  LR: 1e-3  |  Scheduler: WarmupCosine (10 warm-up epochs)
- Optimizer: AdamW (β₁=0.9, β₂=0.999, wd=0.01)
- Grad clip: 1.0


In [ ]:
import pprint
print('Stage 1 full config:')
pprint.pprint(OmegaConf.to_container(stage1_cfg, resolve=True), width=80)


In [ ]:
# Verify multi-GPU and AMP are on (set in default.yaml, inherited by all stage configs)
print(f'multi_gpu : {stage1_cfg.multi_gpu}')
print(f'use_amp   : {stage1_cfg.use_amp}')
print(f'device    : {stage1_cfg.device}')
print(f'num_workers: {stage1_cfg.num_workers}')
# Expected: multi_gpu=True, use_amp=True — if not, edit configs/default.yaml

# Estimate VRAM usage before training
per_gpu_bs = stage1_cfg.train.batch_size // N_GPUS
T          = stage1_cfg.video.max_frames
# Stage 1 only processes pose features, not raw video → very light
pose_mb    = per_gpu_bs * T * POSE_DIM * 2 / 1024**2   # fp16
print(f'\nStage 1 is pose-only (no ViT) — very low VRAM usage')
print(f'  Pose tensor per GPU : {pose_mb:.1f} MB  (batch={per_gpu_bs}, T={T}, fp16)')
print(f'  → batch_size={stage1_cfg.train.batch_size} is safe for 4×24 GB')


In [ ]:
from signx.training.train_stage1 import Stage1Trainer
from signx.pose.feature_compiler  import PoseAwareFeatureCompiler
from signx.pose.pose_extractor    import build_pose_extractor

# Fresh Stage 1 model
stage1_model = make_stage1().to(DEVICE)

pose_extractor = build_pose_extractor(stage1_cfg)
compiler = PoseAwareFeatureCompiler(
    feature_dim       = POSE_DIM,
    frame_dropout     = float(stage1_cfg.get('feature_processing', {}).get('frame_dropout', 0.1)),
    whitening_enabled = bool(stage1_cfg.get('feature_processing',  {}).get('whitening', True)),
).to(DEVICE)

NW = int(stage1_cfg.num_workers)    # 8 = 2 per GPU
BS = int(stage1_cfg.train.batch_size)  # 128

s1_train = DataLoader(train_ds, batch_size=BS, shuffle=True,
                      num_workers=NW, pin_memory=True,
                      collate_fn=collate_video_batch, persistent_workers=(NW>0))
s1_val   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                      num_workers=NW, pin_memory=True,
                      collate_fn=collate_video_batch, persistent_workers=(NW>0))

trainer1 = Stage1Trainer(stage1_model, stage1_cfg,
                         s1_train, s1_val, vocab, pose_extractor, compiler)
print(f'✓  Stage1Trainer ready')
print(f'   DataParallel : {N_GPUS} GPUs  |  AMP : {trainer1.use_amp}')
print(f'   batch={BS}  workers={NW}  epochs={stage1_cfg.train.epochs}')


In [ ]:
# ─── TRAIN STAGE 1 ────────────────────────────────────────────────────────────
# The trainer saves checkpoints to:
#   outputs/checkpoints/stage1_pose2gloss/best.pt
#   outputs/checkpoints/stage1_pose2gloss/epoch???_wer*.pt  (top-5)
# Training curves (JSON) are saved to:
#   outputs/logs/stage1_pose2gloss/curves/
#
# To resume after interruption: re-run this cell — the trainer will pick up
# from the last saved checkpoint (if you add resume logic) or restart.
# ──────────────────────────────────────────────────────────────────────────────

if train_ds:
    gpu_mem()
    try:
        torch.cuda.empty_cache()
        t0 = time.time()
        trainer1.train()
        hrs = (time.time() - t0) / 3600
        print(f'\n✓  Stage 1 complete in {hrs:.2f} h')
    except KeyboardInterrupt:
        print('Training interrupted — checkpoints saved.')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print('⚠️  OOM — try reducing train.batch_size or video.max_frames in stage1_pose2gloss.yaml')
        gpu_mem()
else:
    print('⚠️  train_ds is None — initialise the dataset in Part 3 first.')


In [ ]:
# Plot Stage 1 training curves from saved JSON
import json as _json
from pathlib import Path

def plot_curves(prefix, loss_keys, save_name):
    curve_dir = Path(PROJECT_ROOT) / 'outputs' / 'logs' / prefix / 'curves'
    histories = {}
    for key in loss_keys:
        for fname in [f'{prefix}_{key}.json', f'{key}.json']:
            p = curve_dir / fname
            if p.exists():
                with open(p) as f:
                    histories[key] = _json.load(f)
                break
    if not histories:
        print(f'No curves found in {curve_dir} — run training first.')
        return

    n = len(histories)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (key, vals) in zip(axes, histories.items()):
        ax.plot(vals, lw=1.5)
        ax.set_title(f'{key}')
        ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    plt.suptitle(f'{prefix} Training Curves')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, save_name), dpi=150)
    plt.show()

plot_curves('stage1_pose2gloss', ['loss', 'text', 'word', 'contrast'], 'stage1_curves.png')


---
## Part 6 — Stage 2 Training: Video2Pose

### Goal
Freeze Stage 1. Train a **ViT-B/16 backbone** to predict the same 512-dim latent
from raw RGB video using **MSE loss**.

After Stage 2, we no longer need pose extraction at inference time — the model can
predict pose-like features directly from pixels.

### Why ViT per-frame?
Vision Transformer encodes spatial patch relationships via self-attention, making
it more robust to partial occlusion and signer variation than a plain CNN.

### Multi-GPU & memory
- ViT-B/16 on 224×224 costs ~350 MB per sample (fp32)
- fp16 AMP halves this → ~175 MB per sample per GPU
- Batch 16 per GPU × 4 GPUs = 64 total
- Effective VRAM: 16 × 175 MB ≈ 2.8 GB model weights + 3 GB tensors < 24 GB ✓

### Config summary
- Epochs: 300  |  LR: 8e-4  |  Warm-up: 8 epochs
- Stage 1 is frozen (`freeze_stage1 = true`)


In [ ]:
S1_CKPT = os.path.join(PROJECT_ROOT, 'outputs', 'checkpoints', 'stage1_pose2gloss', 'best.pt')

if not os.path.exists(S1_CKPT):
    print(f'⚠️  Stage 1 checkpoint not found: {S1_CKPT}')
    print('Run Stage 1 training (Part 5) first.')
else:
    from signx.utils import load_checkpoint
    # Re-create a clean Stage 1 model and load weights
    stage1_frozen = make_stage1()
    load_checkpoint(S1_CKPT, stage1_frozen, map_location='cpu', strict=True)
    stage1_frozen.eval()
    for p in stage1_frozen.parameters():
        p.requires_grad = False
    print(f'✓  Stage 1 loaded and frozen from:\n   {S1_CKPT}')
    trainable = sum(p.numel() for p in stage1_frozen.parameters() if p.requires_grad)
    print(f'   Trainable params in frozen S1: {trainable}  (should be 0)')


In [ ]:
print(f'multi_gpu : {stage2_cfg.multi_gpu}')
print(f'use_amp   : {stage2_cfg.use_amp}')

BS2 = int(stage2_cfg.train.batch_size)   # 64 (16/GPU × 4)
NW2 = int(stage2_cfg.num_workers)        # 8

# VRAM estimate for Stage 2
per_gpu_bs2 = BS2 // N_GPUS  # 16 per GPU
vit_mb = per_gpu_bs2 * stage2_cfg.video.max_frames * 3 * 224 * 224 * 2 / 1024**3  # fp16
print(f'\nStage 2 VRAM estimate (fp16):')
print(f'  Video tensors per GPU : {vit_mb:.2f} GB  (batch={per_gpu_bs2})')
print(f'  ViT-B/16 weights      : ~0.35 GB')
print(f'  Total estimate        : ~{vit_mb + 0.35:.1f} GB  / 24 GB available per GPU')
print(f'  → batch_size={BS2} is safe with AMP')


In [ ]:
from signx.training.train_stage2 import Stage2Trainer

# Fresh Stage 2 wraps frozen Stage 1
stage2_model = make_stage2(stage1_frozen).to(DEVICE)

# Stage2Trainer uses pose_extractor + compiler to compute Stage 1 GT latents
# (it calls stage1_model.encode_pose internally via model.stage1_model)
s2_train = DataLoader(train_ds, batch_size=BS2, shuffle=True,
                      num_workers=NW2, pin_memory=True,
                      collate_fn=collate_video_batch, persistent_workers=(NW2>0))
s2_val   = DataLoader(val_ds,   batch_size=BS2, shuffle=False,
                      num_workers=NW2, pin_memory=True,
                      collate_fn=collate_video_batch, persistent_workers=(NW2>0))

pose_extractor2 = build_pose_extractor(stage2_cfg)
compiler2 = PoseAwareFeatureCompiler(
    feature_dim       = POSE_DIM,
    frame_dropout     = 0.0,   # no augmentation for GT latent computation
    whitening_enabled = True,
).to(DEVICE)

trainer2 = Stage2Trainer(stage2_model, stage2_cfg,
                         s2_train, s2_val, pose_extractor2, compiler2)
print(f'✓  Stage2Trainer ready')
print(f'   DataParallel : {N_GPUS} GPUs  |  AMP : {trainer2.use_amp}')
print(f'   batch={BS2}  workers={NW2}  epochs={stage2_cfg.train.epochs}')


In [ ]:
# ─── TRAIN STAGE 2 ────────────────────────────────────────────────────────────
# Checkpoints → outputs/checkpoints/stage2_video2pose/best.pt
# ──────────────────────────────────────────────────────────────────────────────

if train_ds and os.path.exists(S1_CKPT):
    gpu_mem()
    try:
        torch.cuda.empty_cache()
        t0 = time.time()
        trainer2.train()
        hrs = (time.time() - t0) / 3600
        print(f'\n✓  Stage 2 complete in {hrs:.2f} h')
    except KeyboardInterrupt:
        print('Training interrupted.')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print('⚠️  OOM — reduce stage2_cfg train.batch_size or video.max_frames')
        gpu_mem()
else:
    print('⚠️  Skipped — run Stage 1 first.')


In [ ]:
plot_curves('stage2_video2pose', ['loss', 'mse'], 'stage2_curves.png')


---
## Part 7 — Stage 3 Training: Continuous Sign Language Recognition

### Goal
Train the **full CSLR module** on top of frozen Stage 2 video features.
This is the most important stage — it produces gloss sequences from
continuous signing videos.

### Architecture pipeline (per batch)
```
Video (B,T,3,224,224) → Stage2.video2pose → latent (B,T,512)
  → Stage3Model:
      FeaturePruning (512→256 learned mask)
      → TemporalConv  [channels: 512→512, kernels: 5,5, strides: 1,2]
      → BiLSTM        [2 layers, hidden=256, out=512]
      → Transformer   [4-layer enc-dec, d_model=256, heads=8]
      → CTC head      (B, T', 802)
      → Decoder head  (B, L, 802)
```

### Four loss terms
| Loss | λ | Description |
|------|---|-------------|
| CTC | 1.0 | Connectionist Temporal Classification |
| CE  | 1.0 | Cross-entropy on decoder output |
| KD  | 0.5 | KL-divergence: decoder learns from CTC (T=4.0) |
| Reg | 0.1 | L2 latent regulariser keeps features bounded |

### Scheduler: Noam
```
lr = d_model^-0.5 × min(step^-0.5, step × warmup^-1.5)
warmup_steps = 2000
```
The LR rises linearly for 2000 steps then decays as 1/√step.

### Multi-GPU & memory
- Stage 2 ViT is frozen → no grad through ViT
- Only Stage 3 (ResNet+BiLSTM+Transformer) trains
- Effective batch: 16 = 4 per GPU × 4 GPUs
- VRAM per GPU: ~18 GB (ViT forward + Stage3 backward) — within 24 GB ✓


In [ ]:
S2_CKPT = os.path.join(PROJECT_ROOT, 'outputs', 'checkpoints', 'stage2_video2pose', 'best.pt')

if not os.path.exists(S2_CKPT):
    print(f'⚠️  Stage 2 checkpoint not found: {S2_CKPT}')
    print('Run Stage 2 training (Part 6) first.')
else:
    # Load Stage 2 model — its video2pose sub-module will be passed to Stage3Trainer
    stage2_for_s3 = make_stage2()
    from signx.utils import load_checkpoint
    load_checkpoint(S2_CKPT, stage2_for_s3, map_location='cpu', strict=False)
    stage2_for_s3.eval()
    for p in stage2_for_s3.parameters():
        p.requires_grad = False
    stage2_for_s3 = stage2_for_s3.to(DEVICE)

    trainable = sum(p.numel() for p in stage2_for_s3.parameters() if p.requires_grad)
    print(f'✓  Stage 2 loaded and frozen: {S2_CKPT}')
    print(f'   Trainable params: {trainable}  (should be 0)')


In [ ]:
print(f'multi_gpu : {stage3_cfg.multi_gpu}')
print(f'use_amp   : {stage3_cfg.use_amp}')

BS3 = int(stage3_cfg.train.batch_size)    # 16 (4/GPU × 4)
NW3 = int(stage3_cfg.num_workers)         # 8

# VRAM estimate: Stage 2 ViT inference (no grad) + Stage 3 backward
per_gpu_bs3 = BS3 // N_GPUS  # 4
vit_inf_mb  = per_gpu_bs3 * stage3_cfg.video.max_frames * 768 * 2 / 1024**2  # ViT output fp16
print(f'\nStage 3 VRAM estimate (fp16):')
print(f'  ViT output per GPU (no grad) : {vit_inf_mb:.0f} MB  (batch={per_gpu_bs3})')
print(f'  Stage3 model + activations   : ~3-5 GB')
print(f'  → batch={BS3} is safe with 4×24 GB')


In [ ]:
from signx.training.train_stage3 import Stage3Trainer

# Fresh Stage 3 model
stage3_model = make_stage3().to(DEVICE)

s3_train = DataLoader(train_ds, batch_size=BS3, shuffle=True,
                      num_workers=NW3, pin_memory=True,
                      collate_fn=collate_video_batch, persistent_workers=(NW3>0))
s3_val   = DataLoader(val_ds,   batch_size=BS3, shuffle=False,
                      num_workers=NW3, pin_memory=True,
                      collate_fn=collate_video_batch, persistent_workers=(NW3>0))

# Stage3Trainer expects the inner video2pose model (Video2PoseModel),
# NOT the Stage2Model wrapper
video2pose_inner = stage2_for_s3.video2pose

trainer3 = Stage3Trainer(stage3_model, stage3_cfg,
                         s3_train, s3_val, vocab, video2pose_inner)
print(f'✓  Stage3Trainer ready')
print(f'   DataParallel : {N_GPUS} GPUs  |  AMP : {trainer3.use_amp}')
print(f'   batch={BS3}  workers={NW3}  epochs={stage3_cfg.train.epochs}')


In [ ]:
# ─── TRAIN STAGE 3 ────────────────────────────────────────────────────────────
# Checkpoints → outputs/checkpoints/stage3_cslr/best.pt
# Evaluation metric : WER (word error rate, lower is better)
# ──────────────────────────────────────────────────────────────────────────────

if train_ds and os.path.exists(S2_CKPT):
    gpu_mem()
    try:
        torch.cuda.empty_cache()
        t0 = time.time()
        trainer3.train()
        hrs = (time.time() - t0) / 3600
        print(f'\n✓  Stage 3 complete in {hrs:.2f} h')
    except KeyboardInterrupt:
        print('Training interrupted — checkpoints saved.')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print('⚠️  OOM — reduce BS3 (currently {BS3}) or max_frames')
        gpu_mem()
else:
    print('⚠️  Skipped — run Stages 1 and 2 first.')


In [ ]:
# Stage 3 curves — 5 subplots: total + 4 loss components + WER
plot_curves('stage3_cslr', ['loss', 'ctc', 'ce', 'kd', 'reg'], 'stage3_loss_curves.png')

# WER curve
wer_json = Path(PROJECT_ROOT) / 'outputs' / 'logs' / 'stage3_cslr' / 'curves' / 'stage3_cslr_wer.json'
if wer_json.exists():
    with open(wer_json) as f:
        wer_vals = _json.load(f)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(wer_vals, color='crimson', lw=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('WER (%)')
    ax.set_title('Stage 3 — Word Error Rate on Dev Set')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'stage3_wer.png'), dpi=150)
    plt.show()
    print(f'Best WER: {min(wer_vals):.2f}%  @ epoch {wer_vals.index(min(wer_vals))}')


---
## Part 8 — Evaluation & Metrics

### Checkpoint averaging
We average the **top-5 checkpoints** (by validation WER) before evaluation.
This reduces overfitting to specific epochs and typically improves WER by 1-2%.

### Metrics
| Metric | Formula | Better |
|--------|---------|--------|
| WER | (S+D+I) / N | ↓ lower |
| BLEU-1..4 | n-gram precision | ↑ higher |
| PI-Accuracy | \|ref ∩ hyp\| / \|ref\| | ↑ higher |


In [ ]:
from signx.utils.checkpoint import average_checkpoints, load_checkpoint
from signx.inference.evaluate import evaluate_dataset as eval_ds
from signx.training.metrics import compute_wer, compute_bleu, compute_pi_accuracy

S3_CKPT_DIR = Path(PROJECT_ROOT) / 'outputs' / 'checkpoints' / 'stage3_cslr'
S3_BEST     = S3_CKPT_DIR / 'best.pt'
S3_AVG      = S3_CKPT_DIR / 'averaged.pt'

if S3_CKPT_DIR.exists():
    # Collect all epoch checkpoints (exclude best and averaged)
    epoch_ckpts = sorted([
        p for p in S3_CKPT_DIR.glob('epoch*.pt')
        if 'averaged' not in p.name
    ])[-5:]   # top-5 most recent (trainer saves best-K so these are top-K)

    if len(epoch_ckpts) >= 2:
        average_checkpoints(epoch_ckpts, S3_AVG)
        eval_ckpt = str(S3_AVG)
        print(f'✓  Averaged {len(epoch_ckpts)} checkpoints → {S3_AVG}')
    elif S3_BEST.exists():
        eval_ckpt = str(S3_BEST)
        print(f'Using best checkpoint: {S3_BEST}')
    else:
        eval_ckpt = None
        print('⚠️  No Stage 3 checkpoint found.')
else:
    eval_ckpt = None
    print('⚠️  Checkpoint directory not found — run Stage 3 training first.')


In [ ]:
if eval_ckpt and train_ds:
    # Load the averaged weights into a fresh Stage 3 model
    stage3_eval = make_stage3().to(DEVICE)
    load_checkpoint(eval_ckpt, stage3_eval, map_location=DEVICE, strict=False)
    stage3_eval.eval()

    test_loader = DataLoader(test_ds, batch_size=BS3, shuffle=False,
                             num_workers=NW3, pin_memory=True,
                             collate_fn=collate_video_batch)

    # Stage 2 video2pose is already frozen from Part 7
    v2p = video2pose_inner

    print('Evaluating on Dev...')
    dev_metrics  = eval_ds(v2p, stage3_eval, s3_val,   vocab, device=DEVICE)
    print('Evaluating on Test...')
    test_metrics = eval_ds(v2p, stage3_eval, test_loader, vocab, device=DEVICE)

    print(f'\n  {"Dataset":<8} {"Split":<6} {"WER":>8} {"BLEU":>8} {"PI-Acc":>8}')
    print('  ' + '-'*42)
    for split, m in [('Dev', dev_metrics), ('Test', test_metrics)]:
        print(f'  {"OSL":<8} {split:<6} '
              f'{m.get("wer",float("nan")):>8.2f} '
              f'{m.get("bleu",float("nan")):>8.2f} '
              f'{m.get("pi_accuracy",float("nan")):>8.2f}')
else:
    print('Run Stage 3 training first.')
    dev_metrics = test_metrics = {}


In [ ]:
# Inference speed benchmark
if eval_ckpt and train_ds:
    from signx.inference.predict import predict_video
    n_bench = min(20, len(test_ds))
    bench_samples = [test_ds[i] for i in range(n_bench)]

    torch.cuda.synchronize()
    t0 = time.time()
    for s in bench_samples:
        predict_video(s['path'], make_stage2().to(DEVICE), stage3_eval, vocab,
                      stage3_cfg, beam_size=int(stage3_cfg.decode.beam_size), device=DEVICE)
    torch.cuda.synchronize()
    elapsed = time.time() - t0

    print(f'Inference benchmark  (n={n_bench}):')
    print(f'  Total time  : {elapsed:.1f}s')
    print(f'  Videos/sec  : {n_bench/elapsed:.2f}')
    print(f'  ms/video    : {elapsed/n_bench*1000:.0f}')


---
## Part 9 — Qualitative Results & Visualisation

### What we show
1. **Prediction table** — ground truth vs predicted Arabic gloss sequences
2. **t-SNE latent space** — do similar signs cluster together?
3. **Attention heatmap** — which frames does the decoder attend to?


In [ ]:
# Prediction table — 10 test samples
if eval_ckpt and train_ds:
    from signx.inference.predict import predict_video

    n_show = min(10, len(test_ds))
    rows = []
    for i in range(n_show):
        s      = test_ds[i]
        gt_ids = s['gloss_ids'].tolist()
        gt_str = ' '.join(vocab.decode(gt_ids, strip_blank=True))

        pred   = predict_video(s['path'], make_stage2().to(DEVICE), stage3_eval, vocab,
                               stage3_cfg, beam_size=int(stage3_cfg.decode.beam_size), device=DEVICE)
        pred_str = ' '.join(pred)

        # Simple colour: green if correct token, red if not
        match = '✓' if gt_str == pred_str else '✗'
        rows.append((match, gt_str[:50], pred_str[:50]))

    print(f'  {"":<3} {"Ground Truth (Arabic)":<52} {"Prediction":<52}')
    print('  ' + '-'*108)
    for match, gt, pred in rows:
        print(f'  {match}   {gt:<52} {pred:<52}')
else:
    print('Requires Stage 3 checkpoint.')


In [ ]:
# t-SNE of Stage 2 latent features coloured by word class
if eval_ckpt and train_ds:
    try:
        from sklearn.manifold import TSNE
        all_lat, all_ids = [], []
        stage2_tsne = make_stage2().to(DEVICE)
        stage2_tsne.eval()
        with torch.no_grad():
            for i, b in enumerate(s3_val):
                if i >= 30: break
                b     = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in b.items()}
                lat   = stage2_tsne.video2pose(b['videos'])   # (B, T, 512)
                all_lat.append(lat.mean(1).cpu().float().numpy())
                all_ids.extend(b['item_ids'].cpu().tolist())
        all_lat = np.concatenate(all_lat, 0)
        emb = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(all_lat)

        fig, ax = plt.subplots(figsize=(8, 7))
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=all_ids, cmap='tab20', s=15, alpha=0.7)
        plt.colorbar(sc, ax=ax, label='Word ID')
        ax.set_title('t-SNE of Stage 2 Latent Features (Dev set)')
        ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, 'tsne_latent.png'), dpi=150)
        plt.show()
    except ImportError:
        print('scikit-learn not installed: pip install scikit-learn')


---
## Part 10 — Ablation Studies

These experiments measure how much each component contributes.

⚠️  Each ablation re-evaluates the model — expect ~10 min per ablation.
Set `RUN_ABLATIONS = True` to execute. Results are saved to `notebooks/ablation_results.json`.


In [ ]:
RUN_ABLATIONS = False   # 📌 Set True to run

ABLATION_RESULTS = {}

def run_ablation(name, modify_fn=None):
    if not eval_ckpt or not train_ds:
        ABLATION_RESULTS[name] = {'wer': float('nan')}
        return float('nan')
    m = make_stage3().to(DEVICE)
    load_checkpoint(eval_ckpt, m, map_location=DEVICE, strict=False)
    if modify_fn:
        modify_fn(m)
    m.eval()
    metrics = eval_ds(video2pose_inner, m, s3_val, vocab, device=DEVICE)
    ABLATION_RESULTS[name] = metrics
    print(f'  {name:<30} WER={metrics["wer"]:.2f}%')
    return metrics['wer']

if RUN_ABLATIONS:
    print('Running ablations...\n')
    wer_full = run_ablation('Full Pipeline')

    def no_pruning(m):
        m.apply_pruning = lambda x: x   # bypass learned pruning mask
    run_ablation('No Feature Pruning', no_pruning)

    # Print comparison table
    print(f'\n  {"Configuration":<30} {"WER":>8} {"ΔWER":>8}')
    print('  ' + '-'*50)
    for name, m in ABLATION_RESULTS.items():
        delta = m['wer'] - wer_full
        print(f'  {name:<30} {m["wer"]:>8.2f} {delta:>+8.2f}')

    with open(os.path.join(PROJECT_ROOT, 'notebooks', 'ablation_results.json'), 'w') as f:
        _json.dump(ABLATION_RESULTS, f, indent=2, default=str)
    print('\nSaved to notebooks/ablation_results.json')
else:
    print('Set RUN_ABLATIONS = True to run ablation experiments.')


In [ ]:
# Modality ablation (requires full5 pose backend — 5 modalities)
# Shows the contribution of each pose modality to final WER
pose_slices = {
    'MediaPipe':  (0,    258),
    'DWPose':     (258,  657),
    'SMPLer-X':   (657,  1089),
    'PrimeDepth': (1089, 1569),
    'Sapiens':    (1569, 1959),
}

if RUN_ABLATIONS and stage1_cfg.pose.backend == 'full5':
    baseline = ABLATION_RESULTS.get('Full Pipeline', {}).get('wer', float('nan'))
    print(f'  {"Ablated Modality":<20} {"WER":>8} {"ΔWER":>8} {"Rel. Degradation":>18}')
    print('  ' + '-'*60)
    print(f'  {"Full Pipeline":<20} {baseline:>8.2f} {0:>+8.2f} {"—":>18}')
    for mod, (s, e) in pose_slices.items():
        def zero_mod(m, s=s, e=e):
            orig = m.forward
            def fwd(lat, *a, **kw):
                lat = lat.clone(); lat[:,:,s:e] = 0.0; return orig(lat, *a, **kw)
            m.forward = fwd
        wer = run_ablation(f'w/o {mod}', zero_mod)
        rel = (wer - baseline) / (baseline + 1e-9) * 100
        print(f'  {"w/o " + mod:<20} {wer:>8.2f} {wer-baseline:>+8.2f} {rel:>17.1f}%')
else:
    if stage1_cfg.pose.backend != 'full5':
        print('Modality ablation requires pose.backend = full5 in the config.')
    else:
        print('Set RUN_ABLATIONS = True.')


---
## Part 11 — Export & Final Summary

Save all results, print the final performance table, and bundle the model
for inference so anyone can run it with a single command.


In [ ]:
# Consolidated results JSON
results = {
    'model_info': {
        'stage1_params': p1, 'stage2_params': p2, 'stage3_params': p3,
        'total_params': p1 + p2 + p3,
        'vocab_size': VOCAB_SIZE, 'latent_dim': LATENT_DIM, 'pose_dim': POSE_DIM,
        'n_gpus': N_GPUS, 'vram_per_gpu_gb': 24,
    },
    'eval_results': {'dev': dev_metrics, 'test': test_metrics},
    'ablation_results': ABLATION_RESULTS,
    'dataset_stats': split_counts,
}
out_path = os.path.join(PROJECT_ROOT, 'notebooks', 'results.json')
with open(out_path, 'w', encoding='utf-8') as f:
    _json.dump(results, f, indent=2, ensure_ascii=False, default=str)
print(f'✓  Results saved → {out_path}')


In [ ]:
# Final summary banner
print('=' * 65)
print('  SignX — OSL Training Results')
print('=' * 65)
print(f'  Stages trained  : 3-stage pipeline')
print(f'  Hardware        : {N_GPUS} × 24 GB GPU  (fp16 AMP, DataParallel)')
print(f'  Vocab size      : {VOCAB_SIZE} glosses')
print(f'  Latent dim      : {LATENT_DIM}')
print(f'  Pose backend    : {stage1_cfg.pose.backend}  ({POSE_DIM}-dim)')
print(f'  Total params    : {p1+p2+p3:,}')
print()
print(f'  Dev  WER        : {dev_metrics.get("wer",  "—")}')
print(f'  Test WER        : {test_metrics.get("wer", "—")}')
print(f'  Dev  BLEU       : {dev_metrics.get("bleu", "—")}')
print(f'  Test BLEU       : {test_metrics.get("bleu","—")}')
print('=' * 65)


In [ ]:
# Bundle model for inference
import shutil

EXPORT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'export')
os.makedirs(EXPORT_DIR, exist_ok=True)

files_to_copy = {
    'stage3_best.pt'      : eval_ckpt,
    'stage2_best.pt'      : S2_CKPT,
    'gloss_vocab.txt'     : VOCAB_FILE,
}
for dst_name, src in files_to_copy.items():
    dst = os.path.join(EXPORT_DIR, dst_name)
    if src and os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  Copied {dst_name}')
    else:
        print(f'  ⚠️  Source not found: {src}')

from signx.utils import save_config
save_config(stage3_cfg, os.path.join(EXPORT_DIR, 'stage3_config.yaml'))
save_config(stage2_cfg, os.path.join(EXPORT_DIR, 'stage2_config.yaml'))

print(f'\n✓  Export → {EXPORT_DIR}')
print()
print('Run inference with:')
print(f'  python scripts/run_inference.py \\')
print(f'    --video /path/to/video.mp4 \\')
print(f'    --stage2 {EXPORT_DIR}/stage2_best.pt \\')
print(f'    --stage3 {EXPORT_DIR}/stage3_best.pt \\')
print(f'    --vocab  {EXPORT_DIR}/gloss_vocab.txt')
